# Примеры метрик

Каждый блок показывает, что метрика умеет различать и где она слепнет. Аномалия — метка `0`, штатная точка — метка больше `0`. Ряд — кортеж `(время, долгота, широта, метка)`.

Сначала карта исходного трека и пунктирного эталона через аномальный разрыв, затем — разметка модели (две фигуры подряд). Точки синтетические, район Владивостока выбран только чтобы была читаемая подложка. Подложка качается из интернета (по умолчанию схема Carto); тайлы кэшируются в `~/.cache/statistical_researchs/map_tiles`. Без сети подложка подставится из кэша, если он уже был; иначе рисуются только точки.

После правок `plot_sample_track.py` перезапустите ядро или заново выполните первую code-ячейку (`%autoreload` подхватит модуль).

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "app" / "metrics" / "calculate_metrics.py").exists():
        sys.path.insert(0, str(candidate))
        break

import app.metrics.calculate_metrics as calculate_metrics_mod
import app.metrics.plot_sample_track as plot_sample_track_mod

importlib.reload(calculate_metrics_mod)
importlib.reload(plot_sample_track_mod)

CalculateMetrics = calculate_metrics_mod.CalculateMetrics
print_metrics_table = calculate_metrics_mod.print_metrics_table
plot_sample_track = plot_sample_track_mod.plot_sample_track
diagnose_map_tiles = plot_sample_track_mod.diagnose_map_tiles

if not diagnose_map_tiles():
    print("Подложка карт недоступна: проверьте интернет, VPN или выполните pip install certifi")


def course(count=12, step=0.004):
    """Прямой ход на северо-восток. Шаг около 500 м."""
    index = np.arange(count, dtype=float)
    return 43.04 + step * index, 131.87 + step * index


def as_series(latitude, longitude, labels, predicted):
    """Собирает ground truth и prediction на одних координатах."""
    time = np.arange(len(latitude), dtype=float)
    latitude = np.asarray(latitude, dtype=float)
    longitude = np.asarray(longitude, dtype=float)
    ground_truth = (time, longitude, latitude, np.asarray(labels, dtype=float))
    prediction = (time, longitude.copy(), latitude.copy(), np.asarray(predicted, dtype=float))
    return prediction, ground_truth


def show_case(title, prediction, ground_truth, fields, scores=None):
    """Печатает выбранные метрики и рисует трек."""
    metrics = CalculateMetrics.calculate_all_metrics(prediction, ground_truth, scores)
    rows = [(name, name) for name in fields]
    print_metrics_table(metrics, title=title, rows=rows, prefix="  ")
    print()
    plot_sample_track(prediction, ground_truth, title=title)
    return None

## Point-wise F1

Считает каждую точку отдельно: аномалия, найденная не на том индексе, — это и пропуск, и ложная тревога. Время и координаты в формулу не входят.

**Плюс.** Лишняя тревога видна сразу. На участке 4–7 четыре аномалии найдены верно, точка 10 помечена зря. Precision = 4/5 = 0.8: на карте четыре зелёные точки и один оранжевый треугольник.

**Минус.** Тот же участок найден на шаг позже: помечены точки 5–8 вместо 4–7. Point-wise F1 падает до 0.75 из-за одной пропущенной и одной лишней точки. Событие при этом почти на месте — affiliation F1 остаётся около 0.94.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0

found = labels.copy()
found[10] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, found)
show_case(
    "Point-wise: одна лишняя тревога на точке 10",
    prediction,
    ground_truth,
    ["point_precision", "point_recall", "point_f1"],
)

shifted = np.ones(12)
shifted[5:9] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, shifted)
show_case(
    "Point-wise: событие найдено, но на шаг позже",
    prediction,
    ground_truth,
    ["point_precision", "point_recall", "point_f1", "affiliation_f1"],
)

## PR-AUC

Площадь под precision–recall кривой по всем порогам anomaly score. Больший score — более сильная аномалия. Бинарной маски для этой метрики недостаточно.

**Плюс.** Score один и тот же. Порог 0.9 оставляет только две самые сильные аномалии, point-wise F1 = 0.67. Порог 0.5 забирает все четыре, F1 = 1. PR-AUC в обоих случаях равен 1: ранжирование верное и от выбранного порога не зависит.

**Минус.** Тот же идеальный порядок score стоит и на прямом треке, и на петле, которую модель целиком пропустила. PR-AUC снова 1. Геодезический RMSE при этом 0 м против примерно 4.8 км: метрика не видит, куда уехали точки. Если score не передать, `pr_auc` равен `nan`, хотя point-wise F1 по маске считается.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
scores = np.full(12, 0.12)
scores[4:8] = [0.58, 0.74, 0.91, 0.96]
scores[10] = 0.47

for threshold, caption in ((0.9, "жёсткий порог 0.9"), (0.5, "мягкий порог 0.5")):
    predicted = np.where(scores >= threshold, 0.0, 1.0)
    prediction, ground_truth = as_series(latitude, longitude, labels, predicted)
    show_case(
        f"PR-AUC: {caption}",
        prediction,
        ground_truth,
        ["point_f1", "pr_auc"],
        scores=scores,
    )

missed = np.ones(12)
prediction, ground_truth = as_series(latitude, longitude, labels, missed)
without_scores = CalculateMetrics.calculate_all_metrics(prediction, ground_truth)
print(f"Та же мягкая маска без score: pr_auc = {without_scores['pr_auc']}")
print()

show_case(
    "PR-AUC: пропуск на прямой хорде, score идеальный",
    prediction,
    ground_truth,
    ["point_f1", "pr_auc", "geodesic_rmse"],
    scores=scores,
)

loop = latitude.copy()
loop[4:8] += [0.015, 0.045, 0.07, 0.02]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case(
    "PR-AUC: та же модель пропускает петлю",
    prediction,
    ground_truth,
    ["point_f1", "pr_auc", "geodesic_rmse"],
    scores=scores,
)

## Affiliation F1

Сравнивает аномальные интервалы по времени: частичный сдвиг не равен совсем другому событию. Координаты не используются. Precision усредняется только по зонам, куда попало предсказание.

**Плюс.** Сдвиг на один шаг, уже виденный выше. Point-wise F1 = 0.75, affiliation F1 ≈ 0.94. Для affiliation предсказанный интервал почти совпадает с истинным.

**Минус.** Модель помечает аномалией весь трек. Point-wise precision = 4/12 ≈ 0.33 и F1 = 0.5. Affiliation F1 ≈ 0.71: сплошная заливка для неё близка к случайному угадыванию внутри зоны события, а не к полному провалу. Редкость аномалий она штрафует слабее, чем точечная precision.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
shifted = np.ones(12)
shifted[5:9] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, shifted)
show_case(
    "Affiliation: сдвиг на шаг почти не штрафуется",
    prediction,
    ground_truth,
    ["point_f1", "affiliation_precision", "affiliation_recall", "affiliation_f1"],
)

everything = np.zeros(12)
prediction, ground_truth = as_series(latitude, longitude, labels, everything)
show_case(
    "Affiliation: аномалией помечен весь трек",
    prediction,
    ground_truth,
    ["point_precision", "point_f1", "affiliation_precision", "affiliation_recall", "affiliation_f1"],
)

## Detection delay

Время от начала истинного участка до первой точки, которую модель пометила аномалией. Пропуск до конца участка в среднее не входит и не превращается в нулевую задержку. Здесь один шаг ряда — одна единица времени.

**Плюс.** Оба детектора находят 3 из 5 точек, point-wise F1 у обоих 0.75, affiliation тоже совпадает. Ранний начинает с первой точки участка, поздний — только с третьей. Средняя задержка 0 против 2. Эту разницу F1 не видит.

**Минус.** Три отдельных события. Модель сразу ловит первую точку первого и пропускает два других. Средняя задержка равна 0, потому что в неё попало единственное обнаруженное событие. Пропуски лежат в `n_undetected = 2`; point-wise recall при этом 0.11.

In [ ]:
latitude, longitude = course(14)
labels = np.ones(14)
labels[4:9] = 0
early = np.ones(14)
early[4:7] = 0
late = np.ones(14)
late[6:9] = 0
fields = ["point_f1", "affiliation_f1", "detection_delay_mean", "detection_delay_n_undetected"]

prediction, ground_truth = as_series(latitude, longitude, labels, early)
show_case("Delay: детекция с первой точки", prediction, ground_truth, fields)
prediction, ground_truth = as_series(latitude, longitude, labels, late)
show_case("Delay: те же три точки, но с опозданием", prediction, ground_truth, fields)

latitude, longitude = course(18, step=0.0035)
labels = np.ones(18)
labels[2:5] = 0
labels[8:11] = 0
labels[14:17] = 0
only_first = np.ones(18)
only_first[2] = 0
prediction, ground_truth = as_series(latitude, longitude, labels, only_first)
show_case(
    "Delay: мгновенно найдено одно событие из трёх",
    prediction,
    ground_truth,
    ["point_recall", "point_f1", "detection_delay_mean", "detection_delay_n_detected", "detection_delay_n_undetected"],
)

## Geodesic RMSE

Корень из среднего квадрата геодезического отклонения пропущенных аномалий от хорды между соседними штатными точками. Обнаруженная аномалия в сумму не входит.

**Плюс.** Обе модели пропускают один и тот же участок, point-wise F1 у обеих 0. Сдвиг примерно на 200 м даёт RMSE около 222 м. Петля в сторону города — около 4.8 км. Точечная метрика эти пропуски не различает.

**Минус.** Та же петля, но модель помечает её аномалией. RMSE становится 0: штрафуются только false negative. На левой панели петля никуда не делась, пространственного штрафа уже нет.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["point_f1", "geodesic_rmse", "hausdorff_distance"]

small = latitude.copy()
small[4:8] += 0.002
prediction, ground_truth = as_series(small, longitude, labels, missed)
show_case("RMSE: пропущенный сдвиг около 200 м", prediction, ground_truth, fields)

loop = latitude.copy()
loop[4:8] += [0.015, 0.045, 0.07, 0.02]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case("RMSE: пропущена петля в несколько километров", prediction, ground_truth, fields)

prediction, ground_truth = as_series(loop, longitude, labels, labels)
show_case("RMSE: та же петля обнаружена, штраф обнулён", prediction, ground_truth, fields)

## Hausdorff distance

Самое большое геодезическое отклонение среди пропущенных аномальных точек. Это не симметричное расстояние двух множеств, а максимум по тем же false negative, что и у RMSE.

**Плюс.** Четыре пропущенные точки сдвинуты одинаково: RMSE и Хаусдорф совпадают, около 3 км. Если три сдвига малы, а четвёртая точка уходит на 6 км, RMSE остаётся около 3 км, Хаусдорф вырастает до 6 км. Он равен худшей точке, а не среднему.

**Минус.** Худшая точка та же, но в одном треке далеко ушла только она, а в другом — все четыре. Хаусдорф одинаковый, около 6 км. RMSE при этом около 3 км против 6 км. Одна выбросная точка задаёт всю метрику.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["geodesic_rmse", "hausdorff_distance"]

even = latitude.copy()
even[4:8] += 0.027
prediction, ground_truth = as_series(even, longitude, labels, missed)
show_case("Hausdorff: четыре точки сдвинуты одинаково", prediction, ground_truth, fields)

spike = latitude.copy()
spike[4:8] += [0.004, 0.004, 0.004, 0.055]
prediction, ground_truth = as_series(spike, longitude, labels, missed)
show_case("Hausdorff: три точки рядом, одна далеко", prediction, ground_truth, fields)

far = latitude.copy()
far[4:8] += 0.055
prediction, ground_truth = as_series(far, longitude, labels, missed)
show_case("Hausdorff: далеко ушли все четыре точки", prediction, ground_truth, fields)

## Distance loss

Модуль разности длины маршрута, который модель оставила штатным, и длины эталона. `distance_loss_ratio` делит эту разность на длину эталона. Петля, принятая за штатный ход, удлиняет путь сильнее, чем такой же по меткам короткий сдвиг.

**Плюс.** Обе модели пропускают участок целиком, point-wise F1 = 0. Сдвиг почти не меняет длину: отношение около 0.03. Петля, оставленная в штатном маршруте, делает его почти в четыре раза длиннее эталона, отношение около 2.9.

**Минус.** Та же петля обнаружена и выброшена из штатного маршрута. Чёрная линия на правой панели идёт по хорде, distance loss равен 0. Метрика описывает путь, который классификатор счёл штатным, а не исходную геометрию трека.

In [ ]:
latitude, longitude = course()
labels = np.ones(12)
labels[4:8] = 0
missed = np.ones(12)
fields = ["point_f1", "distance_loss", "distance_loss_ratio", "geodesic_rmse"]

bump = latitude.copy()
bump[4:8] += 0.004
prediction, ground_truth = as_series(bump, longitude, labels, missed)
show_case("Distance loss: короткий сдвиг почти не удлиняет путь", prediction, ground_truth, fields)

loop = latitude.copy()
loop[4:8] += [0.02, 0.06, 0.09, 0.03]
prediction, ground_truth = as_series(loop, longitude, labels, missed)
show_case("Distance loss: петля принята за штатный ход", prediction, ground_truth, fields)

prediction, ground_truth = as_series(loop, longitude, labels, labels)
show_case("Distance loss: петля исключена из штатного маршрута", prediction, ground_truth, fields)